# Hash-chat Simples

Este notebook converte uma base de mensagens JSON em uma estrutura de blockchain básica utilizando hashes SHA-256. Abaixo, o código foi dividido em blocos, com explicações detalhadas sobre o que cada parte faz.

### Explicação do Bloco 1: Importações

Neste bloco, importamos as bibliotecas necessárias para o funcionamento do código:
- **`json`**: Utilizada para ler o arquivo de mensagens original e salvar o resultado final no formato JSON.
- **`os`**: Fornece funções para interagir com o sistema operacional, como manipular caminhos de arquivos e diretórios.
- **`hashlib`**: Biblioteca essencial para criptografia. Aqui, ela é usada para gerar os hashes SHA-256 que conectam os blocos da nossa corrente.

In [ ]:
import json
import os
import hashlib

### Explicação do Bloco 2: Carregamento de Dados

Este trecho é responsável por encontrar e ler o arquivo que contém as mensagens originais.
- O código monta o caminho completo para `base_mensagens.json` e tenta abri-lo. Se o arquivo existir, ele carrega o conteúdo para a variável `mensagens`. Adicionamos também uma verificação de segurança: se o arquivo não for encontrado, ele cria algumas mensagens de exemplo para que o restante do código não quebre e você possa testar a lógica.

In [ ]:
# Pega o caminho do diretório atual de execução do notebook
diretorio_atual = os.getcwd()
caminho_arquivo = os.path.join(diretorio_atual, 'base_mensagens.json')

# Verifica se o arquivo existe e carrega os dados
if os.path.exists(caminho_arquivo):
    with open(caminho_arquivo, 'r', encoding='utf-8') as f:
        mensagens = json.load(f)
    print(f"Sucesso: {len(mensagens)} mensagens carregadas do arquivo 'base_mensagens.json'.")
else:
    print(f"Erro: Arquivo '{caminho_arquivo}' não encontrado.")
    print("Criando mensagens de exemplo para que o notebook possa continuar.")
    mensagens = [
        {"id": 1, "sender": "Alice", "timestamp": "2023-01-01 10:00:00", "text": "Olá Bob, tudo bem?"},
        {"id": 2, "sender": "Bob", "timestamp": "2023-01-01 10:05:00", "text": "Oi Alice! Tudo ótimo por aqui."},
        {"id": 3, "sender": "Alice", "timestamp": "2023-01-01 10:10:00", "text": "Que bom! Preciso te contar uma coisa."},
        {"id": 4, "sender": "Bob", "timestamp": "2023-01-01 10:15:00", "text": "Me diga! Estou curioso."},
        {"id": 5, "sender": "Charlie", "timestamp": "2023-01-01 10:20:00", "text": "Cheguei na conversa, sobre o que é?"}
    ]
    print(f"Sucesso: {len(mensagens)} mensagens de exemplo criadas.")


### Explicação do Bloco 3: Inicialização da Corrente Usando SHA-256

Aqui preparamos o terreno para construir o blockchain:
- **`hash_anterior`**: Em um blockchain, cada bloco precisa do hash do bloco anterior. Como o primeiro bloco não tem um "anterior", criamos um hash falso (chamado de Hash Gênesis), composto por 64 zeros.
- **`blockchain`**: Uma lista vazia que irá armazenar os blocos finais gerados.
- Os comandos `print` apenas formatam um cabeçalho de tabela para exibir os resultados de forma organizada na tela.

In [ ]:
# 2. Inicializar a corrente
# O primeiro hash começa com um valor fixo (Hash Gênesis)
hash_anterior = "0" * 64

blockchain = []

print(f"{'ID':<4} | {'MENSAGEM':<30} | {'HASH ATUAL'}")
print("-" * 80)

### Explicação do Bloco 4: Visualização da Geração de Hashes

Este laço `for` percorre todas as mensagens para demonstrar como a corrente é formada:
- Para cada mensagem, ele junta o texto da mensagem com o hash do bloco anterior (`conteudo_para_hash`).
- Em seguida, aplica o algoritmo SHA-256 para gerar um novo hash único (`hash_atual`).
- Ele imprime o ID, um pedaço do texto e os primeiros 10 caracteres do hash gerado.
- O passo mais importante: **`temp_hash_anterior = hash_atual`**. Isso garante que o próximo bloco use o hash que acabou de ser gerado, criando o "elo" da corrente.

In [ ]:
# 3. Percorrer as mensagens e criar a corrente (Apenas visualização)
temp_hash_anterior = hash_anterior # Usamos uma variável temporária para não afetar a montagem final

for msg in mensagens:
    # CONTEÚDO: Texto da mensagem atual + Hash da mensagem anterior
    conteudo_para_hash = msg['text'] + temp_hash_anterior

    # GERAR HASH: SHA-256
    hash_atual = hashlib.sha256(conteudo_para_hash.encode('utf-8')).hexdigest()

    # Mostra os primeiros 10 caracteres do hash
    print(f"{msg['id']:<4} | {msg['text'][:30]:<30} | {hash_atual[:10]}...")

    # O hash atual se torna o 'anterior' para a próxima mensagem do loop
    temp_hash_anterior = hash_atual

print("-" * 80)
print("Corrente de hashes gerada com sucesso!")

### Explicação do Bloco 5: Montagem da Estrutura de Blockchain

Este bloco repete a lógica de geração de hashes do bloco anterior, mas agora com o objetivo de salvar os dados:
- Ele cria um dicionário chamado `bloco` para cada mensagem.
- Esse bloco contém todas as informações originais da mensagem (`id`, `sender`, `timestamp`, `text`) e adiciona duas novas informações cruciais: o `previous_hash` (hash do bloco anterior) e o `hash` (hash do bloco atual).
- Cada bloco gerado é adicionado à lista `blockchain`.

In [ ]:
# 4. Gerar os hashes e montar a nova estrutura
# Reiniciamos o hash_anterior para o valor Gênesis para montar a lista definitiva
hash_anterior = "0" * 64

for msg in mensagens:
    # Conteúdo para o hash: texto + hash anterior
    conteudo_para_hash = msg['text'] + hash_anterior
    hash_atual = hashlib.sha256(conteudo_para_hash.encode('utf-8')).hexdigest()

    # Criar um novo dicionário com os dados originais + o hash atual
    bloco = {
        "id": msg['id'],
        "sender": msg.get('sender', 'Desconhecido'), # Usando .get para evitar erros se a chave não existir
        "timestamp": msg.get('timestamp', 'Sem data'),
        "text": msg['text'],
        "previous_hash": hash_anterior,
        "hash": hash_atual
    }

    # Adicionar à nossa lista de blockchain
    blockchain.append(bloco)

    # Atualizar o hash anterior para a próxima mensagem
    hash_anterior = hash_atual

print(f"Blockchain montado em memória com {len(blockchain)} blocos.")

### Explicação do Bloco 6: Salvamento do Resultado

Finalmente, pegamos a lista `blockchain` (que agora contém todas as mensagens encadeadas com seus respectivos hashes) e a salvamos em um novo arquivo chamado `hashes.json`.
- **`json.dump`**: Converte a lista do Python de volta para o formato JSON.
- **`ensure_ascii=False`**: Garante que caracteres especiais (como acentos) sejam salvos corretamente.
- **`indent=4`**: Formata o arquivo JSON com espaços para que fique legível para humanos.

In [ ]:
# 5. Salvar o resultado em um novo arquivo JSON
caminho_saida = os.path.join(diretorio_atual, 'hashes_sha256.json')

with open(caminho_saida, 'w', encoding='utf-8') as f:
    json.dump(blockchain, f, ensure_ascii=False, indent=4)

print(f"Blocos gerados e salvos com sucesso no arquivo:\n{caminho_saida}")

-------------------------------

### Explicação do Bloco 7: Criação da Corrente de Hashes com SHA-1

Este bloco replica a estrutura do blockchain principal, mas utiliza o algoritmo **SHA-1** no lugar do SHA-256 para fins de comparação.

- **`hashlib.sha1`**: Calcula uma assinatura digital de 160 bits (40 caracteres), permitindo analisar a diferença de tamanho e segurança entre os algoritmos.
- **Encadeamento**: Assim como na versão principal, cada bloco armazena o hash do bloco anterior, garantindo que os dados permaneçam conectados em uma sequência lógica.

In [ ]:
# 6. Gerar blockchain com SHA-1
hash_anterior_sha1 = "0" * 40  # SHA-1 produz 40 caracteres hex

blockchain_sha1 = []

for msg in mensagens:
    conteudo_para_hash = msg['text'] + hash_anterior_sha1
    hash_atual = hashlib.sha1(conteudo_para_hash.encode('utf-8')).hexdigest()

    bloco = {
        "id": msg['id'],
        "sender": msg.get('sender', 'Desconhecido'),
        "timestamp": msg.get('timestamp', 'Sem data'),
        "text": msg['text'],
        "previous_hash": hash_anterior_sha1,
        "hash": hash_atual
    }

    blockchain_sha1.append(bloco)
    hash_anterior_sha1 = hash_atual

print(f"Blockchain SHA-1 montado com {len(blockchain_sha1)} blocos.")

### Explicação do Bloco 8: Exportação dos Dados (SHA-1)

Este bloco é responsável por gravar a estrutura do blockchain gerada na memória em um arquivo físico chamado `hashes_sha1.json`.
- **`os.path.join`**: Cria um caminho de salvamento seguro e compatível com qualquer sistema operacional.
- **`with open`**: Abre (ou cria) o arquivo em modo de escrita, garantindo o fechamento automático após o uso.
- **`json.dump`**: Converte os dados para o formato JSON, preservando a acentuação original (`ensure_ascii=False`) e organizando o texto visualmente para leitura humana (`indent=4`).

In [ ]:
# 7. Salvar o resultado SHA-1 em hashes_sha1.json
caminho_saida_sha1 = os.path.join(diretorio_atual, 'hashes_sha1.json')

with open(caminho_saida_sha1, 'w', encoding='utf-8') as f:
    json.dump(blockchain_sha1, f, ensure_ascii=False, indent=4)

print(f"Arquivo salvo com sucesso:\n{caminho_saida_sha1}")

### Explicação do Bloco 9: Gráfico Matplotlib: Comparando o Comprimento
Este bloco cria uma comparação direta do tamanho do hash gerado, o que reflete seu poder criptográfico.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


sns.set(style="whitegrid")

# Criando um DataFrame
dados_tamanho = pd.DataFrame({
    'Algoritmo': ['SHA-1', 'SHA-256', 'SHA-1', 'SHA-256'],
    'Métrica': ['Bits (Força Bruta)', 'Bits (Força Bruta)', 'Caracteres Físicos', 'Caracteres Físicos'],
    'Valor': [160, 256, 40, 64]
})

# Criando a figura
plt.figure(figsize=(10, 5))

# Usando o barplot do Seaborn, separando as métricas por cor (hue)
ax = sns.barplot(data=dados_tamanho, x='Métrica', y='Valor', hue='Algoritmo', palette=['#EF553B', '#636EFA'])

plt.title("Comparação de Tamanho: SHA-1 vs SHA-256")
plt.ylabel("Quantidade")
plt.xlabel("")

# Adicionando os números em cima das barras
for p in ax.patches:
    ax.annotate(str(p.get_height()), 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 5), 
                textcoords='offset points',
                fontweight='bold')

plt.show()

### Explicação do Bloco 10: Gráfico Plotly: Teste de Aleatoriedade e Distribuição
Este bloco junta todos os hashes de cada corrente gerada e conta a frequência de cada letra e número. O gráfico interativo comprovará que ambos são aparentemente aleatórios (distribuição equilibrada), mas evidenciará o volume brutalmente maior de possibilidades do SHA-256.

In [ ]:
import plotly.express as px
import pandas as pd
from collections import Counter

# 1. Extraindo os hashes gerados
todos_hashes_sha1 = "".join([bloco['hash'] for bloco in blockchain_sha1])
todos_hashes_sha256 = "".join([bloco['hash'] for bloco in blockchain])

# 2. Contando a frequência de cada caractere
contagem_sha1 = Counter(todos_hashes_sha1)
contagem_sha256 = Counter(todos_hashes_sha256)

caracteres_hex = list('0123456789abcdef')
dados_grafico = []

for char in caracteres_hex:
    dados_grafico.append({'Caractere': char, 'Frequência': contagem_sha1.get(char, 0), 'Algoritmo': 'SHA-1'})
    dados_grafico.append({'Caractere': char, 'Frequência': contagem_sha256.get(char, 0), 'Algoritmo': 'SHA-256'})

# 3. Criando o DataFrame (padrão Pandas)
df_freq = pd.DataFrame(dados_grafico)

# 4. Criando o gráfico interativo com Plotly Express
fig = px.bar(df_freq,
             x='Caractere',
             y='Frequência',
             color='Algoritmo',
             barmode='group',
             title='Distribuição de Caracteres Hexadecimais',
             labels={'Frequência': 'Frequência Encontrada', 'Caractere': 'Caractere Hexadecimal'},
             color_discrete_sequence=['#EF553B', '#636EFA'],
             category_orders={"Caractere": caracteres_hex})

# Exibir o gráfico na tela
fig.show()

### Explicação do Bloco 11: Geração e Validação de Hashes (SHA-256)
Este bloco é responsável por criar a corrente de blocos a partir das 5 mensagens exemplo e provar matematicamente que não existem repetições ou colisões entre os hashes gerados.

**`hash_anterior`** = "0" * 64: Cria o "Hash Gênesis", uma sequência de 64 zeros que serve como elo inicial (ponto de partida) para o primeiro bloco da corrente.

**`conteudo_para_hash`** = texto + hash_anterior: Une o texto da mensagem atual com o hash do bloco passado, criando a dependência de dados que garante a segurança da corrente.

**`hashlib.sha256(...).hexdigest()`**: Aplica o algoritmo criptográfico na mensagem concatenada e converte o resultado para o formato de texto hexadecimal legível.

**`lista_hashes.append(hash_atual)`**: Salva cada hash gerado em uma lista na memória para que possam ser contabilizados e verificados no final do processo.

**`set(lista_hashes)`**: Converte a lista final para uma estrutura de conjunto matemático. Como o set do Python remove itens duplicados automaticamente, ele é a ferramenta chave para a prova.

**`len(...)`**: Calcula a quantidade de itens presentes na lista original e no set. A comparação de igualdade entre esses dois tamanhos é o que prova que nenhum hash se repetiu.

In [ ]:
import hashlib
import pandas as pd

# 1. As 5 mensagens de exemplo da base
mensagens = [
    {"id": 1, "text": "Olá Bob, tudo bem?"},
    {"id": 2, "text": "Oi Alice! Tudo ótimo por aqui."},
    {"id": 3, "text": "Que bom! Preciso te contar uma coisa."},
    {"id": 4, "text": "Me diga! Estou curioso."},
    {"id": 5, "text": "Cheguei na conversa, sobre o que é?"}
]

# Inicializando a corrente com o Hash Gênesis
hash_anterior = "0" * 64
dados_tabela = []
lista_hashes = []

# 2. Gerando a corrente de hashes
for msg in mensagens:
    # Concatenando o texto atual com o hash anterior
    conteudo_para_hash = msg['text'] + hash_anterior
    # Gerando o hash SHA-256
    hash_atual = hashlib.sha256(conteudo_para_hash.encode('utf-8')).hexdigest()

    # Salvando os dados para construir a tabela
    dados_tabela.append({
        "ID": msg['id'],
        "Mensagem": msg['text'],
        "Hash SHA-256": hash_atual
    })

    lista_hashes.append(hash_atual)
    hash_anterior = hash_atual  # O hash atual vira o anterior da próxima iteração

# 3. Demonstrando a tabela de hashes gerados
print("TABELA DE HASHES GERADOS:")
print("-" * 120)
df_tabela = pd.DataFrame(dados_tabela)
print(df_tabela.to_string(index=False))
print("-" * 120)

# 4. Comprovando que não há repetições
total_hashes = len(lista_hashes)
# Se o tamanho do set for igual ao tamanho da lista, todos são únicos.
hashes_unicos = len(set(lista_hashes))

print(f"\nResumo da Verificação de Repetições:")
print(f"Total de blocos gerados: {total_hashes}")
print(f"Total de hashes únicos:  {hashes_unicos}")

if total_hashes == hashes_unicos:
    print("✓ Resultado: Não há repetições. A integridade estrutural e unicidade estão validadas.")
else:
    print("⚠ Erro: Foram encontradas colisões/repetições de hash.")